# 23｜強化學習_策略價值與深度RL

以兩臂 bandit 示範探索、回饋與增量式價值估計。

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

## 投影片與程式的閱讀方式

對應 `slides/23_強化學習_策略價值與深度RL.md`，目前 43 頁。逐圖完整表見 `programs/SLIDE_NOTEBOOK_MAP_05_25.md`。

回報/Q 手算、LineWorld/Bellman/DQN、policy gradient/PPO、影格介面。

各格會標示實驗資料、評估方式與適用範圍；架構圖用於說明機制與張量形狀。

| 程式代碼 | 投影片頁次 | 本格學習內容 |
|---|---|---|
| 23-03 | 4、5、24 | 課堂小實驗 |
| 23-04 | 3、24、39 | 執行結果圖 |
| 23-05 | 4、5、7、8、9、10、11、12、13、16、21、22、23、40、43 | 1. 回報、Gym 形式與 CartPole 觀測 |
| 23-06 | 16、17、18、19、20、24、25、26、27、28、29、36、37、40、43 | 2. Bellman、表格 Q 與小型 DQN 更新 |
| 23-07 | 6、14、15、30、31、32、33、34、35、36、37、38、39、41、42、43 | 3. REINFORCE、actor–critic、PPO 與影格介面 |

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
# 程式 23-01｜投影片 23：2 頁
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])

In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
# 程式 23-02｜投影片 23：2 頁
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
if ROOT.name == "notebooks" and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    svg_path = folder / f"{name}.svg"
    plt.gcf().savefig(svg_path, bbox_inches="tight", metadata={"Date": None})
    svg_path.write_text("\n".join(line.rstrip() for line in svg_path.read_text().splitlines()) + "\n")
    plt.show()
    plt.close(plt.gcf())

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")

rng = np.random.default_rng(SEED)

## 課堂小實驗

Bandit 沒有下一狀態與延後回報；不能用它代替 MDP、Q-learning 或 DQN。單次軌跡也不是多種子成效證據。下方補齊有狀態的環境與回報。

In [ ]:
# 程式 23-03｜投影片 23：4、5、24 頁
true_means = np.array([0.2, 0.8])
Q = np.zeros(2); counts = np.zeros(2, dtype=int); rewards = []
epsilon = 0.12
for step in range(300):
    action = rng.integers(2) if rng.random() < epsilon else int(np.argmax(Q))
    reward = rng.normal(true_means[action], 0.25)
    counts[action] += 1
    Q[action] += (reward - Q[action]) / counts[action]
    rewards.append(reward)
pd.DataFrame({'arm': [0,1], 'pulls': counts, 'estimated_value': Q, 'true_mean': true_means}).round(3)

## 執行結果圖

In [ ]:
# 程式 23-04｜投影片 23：3、24、39 頁
rewards = np.array(rewards)
moving = np.convolve(rewards, np.ones(25)/25, mode='valid')
plt.plot(np.arange(24, len(rewards)), moving)
plt.axhline(true_means.max(), color='tab:red', ls='--', label='best arm mean')
plt.xlabel('interaction'); plt.ylabel('25-step mean reward'); plt.title('Exploration gradually identifies the better action')
plt.legend()
savefig('23_rl_demo')
report('reinforcement_learning', {'counts': counts.tolist(), 'Q': Q.tolist(), 'mean_reward': rewards.mean()})

## 1. 回報、Gym 形式與 CartPole 觀測

內嵌 LineWorld 不依賴 Gym；reset/step 回傳欄位與教材同角色。CartPole 圖只對應四維觀測與幾何，不執行或聲稱重現其物理模擬。

In [ ]:
# 程式 23-05｜投影片 23：4、5、7、8、9、10、11、12、13、16、21、22、23、40、43 頁
def returns23(rewards,gamma):
    out=np.zeros(len(rewards));acc=0.
    for i in range(len(rewards)-1,-1,-1):acc=rewards[i]+gamma*acc;out[i]=acc
    return out
assert np.allclose(returns23([10,0,-50],.8),[-22,-40,-50])
def q_update23(q,r,next_q,terminated=False):return q+.1*(r+.9*(not terminated)*next_q-q)
assert np.allclose([q_update23(.5,0,.8),q_update23(.2,1,0,True),q_update23(.1,-.1,.4)],[.522,.28,.116])
class LineWorld23:
    def reset(self):self.state=0;self.steps=0;return self.state,{}
    def step(self,action):
        self.state=int(np.clip(self.state+(1 if action else -1),0,3));self.steps+=1
        terminated=self.state==3;truncated=self.steps>=8 and not terminated
        return self.state,(1. if terminated else -.05),terminated,truncated,{}
env23=LineWorld23();state23,_=env23.reset();trajectory23=[]
for action23 in [1,1,1]:
    nxt23,r23,term23,trunc23,_=env23.step(action23);trajectory23.append((state23,action23,r23,nxt23,term23,trunc23));state23=nxt23
assert trajectory23[-1][4] and not trajectory23[-1][5]
# 一般外部時間限制仍可 bootstrap，真正 terminal 則不能
print('Terminal/truncated TD targets:',1.,-.05+.9*.8)
fig,axes=plt.subplots(1,2,figsize=(10,3.5));axes[0].bar(np.arange(3),returns23([10,0,-50],.8));axes[0].set(xlabel='time',ylabel='return',title='Delayed negative credit')
from matplotlib.patches import Rectangle
obs23=np.array([.3,.1,.2,-.05]);cart_x23=obs23[0];theta23=obs23[2]
axes[1].add_patch(Rectangle((cart_x23-.2,-.1),.4,.2,color='gray'));axes[1].plot([cart_x23,cart_x23+np.sin(theta23)],[0,np.cos(theta23)],lw=5)
axes[1].set(xlim=(-1,1),ylim=(-.2,1.2),aspect='equal',title='CartPole observation geometry');savefig('c23_return_observation')
print('CartPole state roles: x, x_velocity, angle, angular_velocity:',obs23)

## 2. Bellman、表格 Q 與小型 DQN 更新

四狀態 LineWorld 的已知轉移可做 value iteration；採樣互動則用 Q-learning。DQN 以共享 MLP 近似 Q，replay 抽樣並定期複製 target network。這是小環境 CPU 實驗，非 CartPole／Atari 基準。

In [ ]:
# 程式 23-06｜投影片 23：16、17、18、19、20、24、25、26、27、28、29、36、37、40、43 頁
gamma23=.9;Qstar23=np.zeros((4,2))
for _ in range(100):
    new23=np.zeros_like(Qstar23)
    for s23 in range(3):
        for a23 in range(2):
            ns23=int(np.clip(s23+(1 if a23 else -1),0,3));terminal23=ns23==3;reward23=1. if terminal23 else -.05
            new23[s23,a23]=reward23+gamma23*(not terminal23)*Qstar23[ns23].max()
    Qstar23=new23
assert np.all(Qstar23[:3].argmax(1)==1)
rng23=np.random.default_rng(SEED);table23=np.zeros((4,2));buffer23=[]
for episode23 in range(100):
    s23,_=env23.reset()
    while True:
        a23=int(rng23.integers(2)) if rng23.random()<.3 else int(table23[s23].argmax())
        ns23,r23,term23,trunc23,_=env23.step(a23)
        table23[s23,a23]+=.2*(r23+.9*(not term23)*table23[ns23].max()-table23[s23,a23])
        buffer23.append((s23,a23,r23,ns23,term23));s23=ns23
        if term23 or trunc23:break
# one-hot state -> hidden -> two action values
params23=[rng23.normal(0,.2,(4,12)),np.zeros(12),rng23.normal(0,.2,(12,2)),np.zeros(2)]
target23=[p.copy() for p in params23];losses23=[]
def forward23(states,params):
    onehot=np.eye(4)[states];h=np.maximum(onehot@params[0]+params[1],0);return h@params[2]+params[3],h,onehot
for step23 in range(250):
    items23=[buffer23[i] for i in rng23.integers(len(buffer23),size=32)]
    states23,actions23,reward23,next_states23,done23=map(np.array,zip(*items23))
    target_q23,_,_=forward23(next_states23,target23);td23=reward23+.9*(~done23)*target_q23.max(1)
    q23,h23,oh23=forward23(states23,params23);error23=q23[np.arange(32),actions23]-td23;losses23.append(np.mean(error23**2))
    dq23=np.zeros_like(q23);dq23[np.arange(32),actions23]=2*error23/32;dh23=(dq23@params23[2].T)*(h23>0)
    gradients23=[oh23.T@dh23,dh23.sum(0),h23.T@dq23,dq23.sum(0)]
    for p23,g23 in zip(params23,gradients23):p23-=.05*g23
    if (step23+1)%25==0:target23=[p.copy() for p in params23]
fig,axes=plt.subplots(1,2,figsize=(10,3.5));axes[0].imshow(Qstar23[:3],cmap='viridis');axes[0].set(xlabel='action: left/right',ylabel='state',title='Bellman optimal Q')
axes[1].plot(losses23);axes[1].set(xlabel='replay update',ylabel='TD MSE');savefig('c23_bellman_dqn')
print('Optimal/tabular/MLP greedy actions:',Qstar23[:3].argmax(1),table23[:3].argmax(1),forward23(np.arange(3),params23)[0].argmax(1))
assert np.isfinite(losses23).all()

## 3. REINFORCE、actor–critic、PPO 與影格介面

策略梯度用已取樣動作的 log probability 乘回報／優勢；critic 估計回報。以下驗算方向和 PPO clipping，沒有把短算式稱為完整 PPO 訓練。

In [ ]:
# 程式 23-07｜投影片 23：6、14、15、30、31、32、33、34、35、36、37、38、39、41、42、43 頁
softmax23=lambda z:np.exp(z-np.max(z))/np.exp(z-np.max(z)).sum()
logits23=np.array([0.,0.]);prob23=softmax23(logits23);chosen23=1;advantage23=2.
grad_loss23=advantage23*(prob23-np.eye(2)[chosen23]);updated23=logits23-.1*grad_loss23
assert softmax23(updated23)[chosen23]>prob23[chosen23]
critic23=np.array([.3,.4]);returns_target23=np.array([1.,-.2]);advantages23=returns_target23-critic23
actor_loss23=-np.mean(np.log(np.array([.7,.4]))*advantages23);critic_loss23=np.mean((critic23-returns_target23)**2)
ratios23=np.linspace(.5,1.5,100);fig,ax=plt.subplots()
for advantage23 in [1.,-1.]:ax.plot(ratios23,np.minimum(ratios23*advantage23,np.clip(ratios23,.8,1.2)*advantage23),label=f'advantage={advantage23:g}')
ax.set(xlabel='new / old action probability',ylabel='clipped objective');ax.legend();savefig('c23_ppo_clip')
print('Actor loss / critic MSE:',actor_loss23,critic_loss23)
# 合成影格：max over consecutive frames -> grayscale -> resize -> stack
frames23=rng23.integers(0,256,size=(5,210,160,3),dtype=np.uint8);stack23=[]
for t23 in range(1,5):
    max_frame23=np.maximum(frames23[t23-1],frames23[t23]);gray23=max_frame23@np.array([.299,.587,.114])
    rows23=np.linspace(0,209,84).astype(int);cols23=np.linspace(0,159,84).astype(int);stack23.append(gray23[rows23[:,None],cols23[None,:]]/255.)
stack23=np.stack(stack23);assert stack23.shape==(4,84,84)
plt.imshow(stack23[0],cmap='gray');plt.title('Synthetic grayscale frame: shape check only');savefig('c23_frame_preprocessing')
# 多種子 bandit 評估，報樣本數與平均/標準差；不據單次曲線推優越性
means23=[]
for seed23 in range(5):
    rg23=np.random.default_rng(seed23);q23=np.zeros(2);counts23=np.zeros(2);rewards23=[]
    for _ in range(300):
        a23=int(rg23.integers(2)) if rg23.random()<.12 else int(q23.argmax());r23=rg23.normal([.2,.8][a23],.25)
        counts23[a23]+=1;q23[a23]+=(r23-q23[a23])/counts23[a23];rewards23.append(r23)
    means23.append(np.mean(rewards23[-100:]))
print('Five-seed last-100 training reward mean/std:',np.mean(means23),np.std(means23,ddof=1))

## 練習：改動一個參數

先預測結果，再修改程式中的一個參數並重跑；比較圖表或數值，說明差異。